# Component 3 - XAI step 3: which pixels make StarDist see a cell?

**Seg-Grad-CAM** (Vinogradova et al. 2020) for single cells: the explained score is the sum of StarDist's object
probability over the cell; Grad-CAM weighting at StarDist's `features` layer (the last layer before the outputs).

* Cells: a random sample of 30 GREEN and 30 YELLOW / RED cells from the 40 annotated crops, each predicted by the
  Phase 2 fold model that never saw it (the same cells and levels as step 2).
* **Primary (faithfulness):** remove the 20 % hottest pixels around the cell; the score must drop more than when
  the same mask is shifted elsewhere (Wilcoxon, p < 0.05).
* Secondary: heat on the cell + its walls vs its area share; wrong vs correct cells.

Rules fixed in `config.yaml -> xai.segmentation` before any result. Nothing here changes a number.

**GPU runtime** (Runtime > Change runtime type > T4 GPU).


In [ ]:
# 1. Setup (mount Drive, pull latest code, copy images, set paths)
from google.colab import drive; drive.mount("/content/drive")
REPO_DIR = "/content/drive/MyDrive/Colab Notebooks/Research_Project/xai-cornea-grading"
!git -C "{REPO_DIR}" checkout -- ml/cellular/notebooks/
!git -C "{REPO_DIR}" pull --ff-only
%run "{REPO_DIR}/ml/cellular/colab_setup.py"

In [ ]:
# 2. Install StarDist (not in Colab by default) + unit tests
!pip -q install tensorflow tf-keras stardist
!python -m pytest tests/test_seg_gradcam.py -q


In [ ]:
# 3. Seg-Grad-CAM evaluation on the 40 crops + example figure (GPU, ~5 min)
!python -m xai.seg_explain --evaluate 2>&1 | grep -v -i warning


In [ ]:
# 4. Show the example cells
from IPython.display import Image, display
display(Image(f"{OUTPUT_ROOT}/xai/segmentation/figures/examples.png", width=1100))
